# 28 Patterns across columns: components, clusters, distances, maps, log ratios, profiles, factors

Real-data checks of the group "Patterns across columns". This notebook holds only simplydata: data read with
`sdata.table`, the words, the numbers read back with simplydata's own words (through `Result`), and the comparison with
`references/28_patterns_across_columns.csv`; each comparison prints its "Source:" with a link.

References: printed where a printed number exists (Rosseel 2012 for the political democracy model); otherwise the
help-page example of the library simplydata calls, run once by `scripts/references/28_patterns_across_columns.py` (R
prcomp, kmeans, hclust, cluster, vegan, zCompositions, compositions, MASS lda, psych). Signs of components, axes and
loadings are arbitrary, so each column is compared after taking the sign of the reference; correspondence axes are
also scaled to length 1 (programs scale CA scores differently); clusters are compared up to their numbering. Tolerances:
relative 1e-10 for closed formulas and decompositions, 1e-6 for iterative fits, permutation p-values within 4 Monte
Carlo standard errors, printed values to half a unit of their last digit.

In [1]:
import simplydata as sdata

from reference_checks import Comparisons, Result, quiet

cmp = Comparisons("28_patterns_across_columns")
DATA = "../data/"
FETCHED = "../data/fetched/"  # the datasets this repository does not host (written by scripts/get_data.py)

## 1. Principal components: USArrests (`prcomp(USArrests, scale. = TRUE)`, the help-page example)

In [2]:
COLS = ["Murder", "Assault", "UrbanPop", "Rape"]
arrests = sdata.table(cmp.data_file("USArrests"))
pcs = Result(quiet(arrests.find_principal_components_of, COLS))
cmp.check_vector("USArrests", "find_principal_components_of", "standard deviations", pcs["standard deviation"])
cmp.check_vector("USArrests", "find_principal_components_of", "spread explained (%)", pcs["spread explained (%)"])
cmp.check_matrix("USArrests", "find_principal_components_of", "weights", list(zip(*pcs.columns_of([f"weight of {c}" for c in COLS]))))
scores = sdata.table(cmp.data_file("USArrests"))
quiet(scores.add_principal_components, COLS, how_many=2)
cmp.check_matrix("USArrests", "add_principal_components (how_many=2)", "scores", Result(scores).columns_of(["PC1", "PC2"]))

Read "USArrests.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  USArrests | find_principal_components_of | standard deviations [1]: simplydata 1.57487827439, R stats prcomp (help page example) (stats 4.6.2) 1.57487827439; relative difference 2.82e-16 (allowed 1e-10)
      Source: R stats prcomp (help page example) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/prcomp.html
agrees  USArrests | find_principal_components_of | standard deviations [2]: simplydata 0.994869414818, R stats prcomp (help page example) (stats 4.6.2) 0.994869414818; relative difference 2.23e-16 (allowed 1e-10)
      Source: R stats prcomp (help page example) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/prcomp.html
agrees  USArrests | find_principal_components_of | standard deviations [3]: simplydata 0.597129115503, R stats prcomp (help page example) (stats 4.6.2) 0.597129115503; relative difference 3.7

## 2. Clusters

k-means on the standardised USArrests (R `kmeans(nstart = 25)` after the same `set.seed`), hierarchical clustering
with Ward's linkage (R `hclust(method = "ward.D2")`), silhouette widths (R `cluster::silhouette`), and Bray-Curtis
with average linkage on varespec (R `hclust(vegdist(varespec), "average")`).

In [3]:
def clusters(table, columns, **options):
    t = sdata.table(table)
    quiet(t.add_clusters, columns, **options)
    return Result(t)["cluster"]


for seed in (1, 2):
    cmp.check_vector("USArrests", f'add_clusters (test_method="k-means", how_many=4, with_seed={seed})', "cluster",
                     clusters(cmp.data_file("USArrests"), COLS, how_many=4, with_seed=seed))
cmp.check_vector("USArrests", 'add_clusters (test_method="hierarchical", linkage="ward", how_many=4)', "cluster",
                 clusters(cmp.data_file("USArrests"), COLS, how_many=4, test_method="hierarchical", linkage="ward"))
silhouette = Result(quiet(sdata.table(cmp.data_file("USArrests")).add_clusters, COLS, how_many=4, show="silhouette"))
cmp.check_vector("USArrests", 'add_clusters (how_many=4, show="silhouette")', "silhouette widths", silhouette["silhouette width"])
varespec = sdata.table(cmp.data_file("varespec"))
SPECIES = [c for c in quiet(varespec.get_column_names) if c != "row"]
cmp.check_vector("varespec (vegan)", 'add_clusters (test_method="hierarchical", measure="bray-curtis", how_many=3)', "cluster",
                 clusters(cmp.data_file("varespec"), SPECIES, how_many=3, test_method="hierarchical", measure="bray-curtis"))

Read "USArrests.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  USArrests | add_clusters (test_method="k-means", how_many=4, with_seed=1) | cluster: rows in the same group: simplydata 50, R stats kmeans (nstart = 25, same seed) (stats 4.6.2) 50; difference 0.00e+00 (allowed 0e+00)
      Source: R stats kmeans (nstart = 25, same seed) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/kmeans.html
Read "USArrests.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  USArrests | add_clusters (test_method="k-means", how_many=4, with_seed=2) | cluster: rows in the same group: simplydata 50, R stats kmeans (nstart = 25, same seed) (stats 4.6.2) 50; difference 0.00e+00 (allowed 0e+00)
      Source: R stats kmeans (nstart = 25, same seed) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/kmeans.html
Read "USArrests.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  USArrests | add_clusters (test_method="hierarchical"

agrees  varespec (vegan) | add_clusters (test_method="hierarchical", measure="bray-curtis", how_many=3) | cluster: rows in the same group: simplydata 24, R hclust (average) on vegan vegdist (vegan 2.7.8) 24; difference 0.00e+00 (allowed 0e+00)
      Source: R hclust (average) on vegan vegdist (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/vegdist.html


## 3. Distances between rows

Bray-Curtis and Jaccard (presence) on varespec (the help-page data of `vegan::vegdist`); Aitchison on the dune counts
(the 27 species with at least two values above 0), against `zCompositions::cmultRepl(method = "GBM")` + `vegdist`.

In [4]:
for measure in ("bray-curtis", "jaccard"):
    d = Result(quiet(varespec.find_distances_between_rows, SPECIES, measure=measure))
    cmp.check_matrix("varespec (vegan)", f'find_distances_between_rows (measure="{measure}")', "distances", d.columns_of(d.columns[1:]))
dune = sdata.table(cmp.data_file("dune"))
DUNE_ALL = list(quiet(dune.get_column_names))
DUNE = Result(sdata.table(cmp.data_file("DUNE_27_species")))["species"]
d = Result(quiet(dune.find_distances_between_rows, DUNE, measure="aitchison"))
cmp.check_matrix("dune (vegan), 27 species", 'find_distances_between_rows (measure="aitchison")', "distances", d.columns_of(d.columns[1:]))

agrees  varespec (vegan) | find_distances_between_rows (measure="bray-curtis") | distances, column 1 [1]: simplydata 0, R vegan vegdist (help page data) (vegan 2.7.8) 0; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan vegdist (help page data) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/vegdist.html
agrees  varespec (vegan) | find_distances_between_rows (measure="bray-curtis") | distances, column 1 [2]: simplydata 0.531002122668, R vegan vegdist (help page data) (vegan 2.7.8) 0.531002122668; relative difference 2.09e-16 (allowed 1e-10)
      Source: R vegan vegdist (help page data) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/vegdist.html
agrees  varespec (vegan) | find_distances_between_rows (measure="bray-curtis") | distances, column 1 [3]: simplydata 0.668066081457, R vegan vegdist (help page data) (vegan 2.7.8) 0.668066081457; relative difference 1.66e-16 (allowed 1e-10)
      Source:

agrees  dune (vegan), 27 species | find_distances_between_rows (measure="aitchison") | distances, column 1 [1]: simplydata 0, R zCompositions cmultRepl (GBM) + vegan vegdist (aitchison) (zCompositions 1.6.2) 0; relative difference 0.00e+00 (allowed 1e-10)
      Source: R zCompositions cmultRepl (GBM) + vegan vegdist (aitchison) (zCompositions 1.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/zCompositions/html/cmultRepl.html
agrees  dune (vegan), 27 species | find_distances_between_rows (measure="aitchison") | distances, column 1 [2]: simplydata 3.64818788331, R zCompositions cmultRepl (GBM) + vegan vegdist (aitchison) (zCompositions 1.6.2) 3.64818788331; relative difference 0.00e+00 (allowed 1e-10)
      Source: R zCompositions cmultRepl (GBM) + vegan vegdist (aitchison) (zCompositions 1.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/zCompositions/html/cmultRepl.html
agrees  dune (vegan), 27 species | find_distances_between_rows (measure="aitchison")

## 4. Maps of the rows: NMDS, principal coordinates, correspondence axes

`add_nmds` (R `vegan::metaMDS` with the same seed), `add_principal_coordinates(how_many=3)` (R `cmdscale(vegdist())`),
`add_correspondence_axes(how_many=3)` (R `vegan::cca(dune)`, each axis at length 1).

In [5]:
for seed, tries in ((1, 20), (5, 30)):
    m = sdata.table(cmp.data_file("varespec"))
    quiet(m.add_nmds, SPECIES, tries=tries, with_seed=seed)
    cmp.check_matrix("varespec (vegan)", f"add_nmds (tries={tries}, with_seed={seed})", "map", Result(m).columns_of(["NMDS1", "NMDS2"]))
p = sdata.table(cmp.data_file("varespec"))
quiet(p.add_principal_coordinates, SPECIES, how_many=3)
cmp.check_matrix("varespec (vegan)", 'add_principal_coordinates (measure="bray-curtis", how_many=3)', "coordinates",
                 Result(p).columns_of(["PCo1", "PCo2", "PCo3"]))
c = sdata.table(cmp.data_file("dune"))
quiet(c.add_correspondence_axes, DUNE_ALL, how_many=3)
cmp.check_matrix("dune (vegan)", "add_correspondence_axes (how_many=3)", "row axes", Result(c).columns_of(["CA1", "CA2", "CA3"]))

Read "varespec.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  varespec (vegan) | add_nmds (tries=20, with_seed=1) | map, column 1 [1]: simplydata -0.219892344602, R vegan metaMDS (same seed) (vegan 2.7.8) -0.219892344602; relative difference 3.79e-16 (allowed 1e-06)
      Source: R vegan metaMDS (same seed) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/metaMDS.html
agrees  varespec (vegan) | add_nmds (tries=20, with_seed=1) | map, column 1 [2]: simplydata 0.490944207504, R vegan metaMDS (same seed) (vegan 2.7.8) 0.490944207504; relative difference 1.13e-16 (allowed 1e-06)
      Source: R vegan metaMDS (same seed) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/metaMDS.html
agrees  varespec (vegan) | add_nmds (tries=20, with_seed=1) | map, column 1 [3]: simplydata 0.80732238834, R vegan metaMDS (same seed) (vegan 2.7.8) 0.80732238834; relative difference 0.00e+00 (allowed 1e

agrees  varespec (vegan) | add_principal_coordinates (measure="bray-curtis", how_many=3) | coordinates, column 1 [1]: simplydata 0.0945937305475, R cmdscale on vegan vegdist (stats 4.6.2) 0.0945937305475; relative difference 4.40e-16 (allowed 1e-10)


      Source: R cmdscale on vegan vegdist (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/cmdscale.html
agrees  varespec (vegan) | add_principal_coordinates (measure="bray-curtis", how_many=3) | coordinates, column 1 [2]: simplydata -0.312488090075, R cmdscale on vegan vegdist (stats 4.6.2) -0.312488090075; relative difference 1.78e-16 (allowed 1e-10)
      Source: R cmdscale on vegan vegdist (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/cmdscale.html
agrees  varespec (vegan) | add_principal_coordinates (measure="bray-curtis", how_many=3) | coordinates, column 1 [3]: simplydata -0.351065072382, R cmdscale on vegan vegdist (stats 4.6.2) -0.351065072382; relative difference 1.58e-16 (allowed 1e-10)
      Source: R cmdscale on vegan vegdist (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/cmdscale.html
agrees  varespec (vegan) | add_principal_coordinates (measure="bray-curtis",

## 5. Axes that separate groups (linear discriminant analysis): iris (the help-page data of `MASS::lda`)

In [6]:
iris = sdata.table(DATA + "iris.csv")
quiet(iris.add_axes_that_separate_groups, columns=["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"], for_each_group_in="Species",
      how_many=2)
cmp.check_matrix("iris", "add_axes_that_separate_groups (how_many=2)", "LD scores", Result(iris).columns_of(["LD1", "LD2"]))

Read "iris.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  iris | add_axes_that_separate_groups (how_many=2) | LD scores, column 1 [1]: simplydata 8.061799783, R MASS lda (help page data) (MASS 7.3.66) 8.061799783; relative difference 2.20e-16 (allowed 1e-10)
      Source: R MASS lda (help page data) (MASS 7.3.66) (run reference) - https://search.r-project.org/CRAN/refmans/MASS/html/lda.html
agrees  iris | add_axes_that_separate_groups (how_many=2) | LD scores, column 1 [2]: simplydata 7.1286877207, R MASS lda (help page data) (MASS 7.3.66) 7.1286877207; relative difference 0.00e+00 (allowed 1e-10)
      Source: R MASS lda (help page data) (MASS 7.3.66) (run reference) - https://search.r-project.org/CRAN/refmans/MASS/html/lda.html
agrees  iris | add_axes_that_separate_groups (how_many=2) | LD scores, column 1 [3]: simplydata 7.48982797134, R MASS lda (help page data) (MASS 7.3.66) 7.48982797134; relative difference 0.00e+00 (allowed 1e-10)
      Sou

## 6. Log ratios

clr, alr (against Cl) and ilr on Hydrochem (the help-page data of `compositions::clr`; no zeros), against R
compositions (every 10th of its 485 rows); on the dune counts with zeros, the default GBM zero replacement and
`zeros_replaced_by="adding 1"`.

In [7]:
PARTS = ["H", "Na", "K", "Mg", "Ca", "Sr", "Ba", "NH4", "Cl", "NO3", "PO4", "SO4", "HCO3", "TOC"]
for test_method, extra, prefix in (("clr", {}, "clr_"), ("alr", {"against": "Cl"}, "alr_"), ("ilr", {}, "ilr")):
    h = sdata.table(cmp.data_file("Hydrochem"))
    quiet(h.add_log_ratio_columns, PARTS, test_method=test_method, **extra)
    out = Result(h)
    label = f'add_log_ratio_columns (test_method="{test_method}"' + (', against="Cl")' if extra else ")")
    cmp.check_matrix("Hydrochem (compositions)", label, "log ratios", out.columns_of([c for c in out.columns if c.startswith(prefix)]))
for replaced, label in (({}, "estimated small values"), ({"zeros_replaced_by": "adding 1"}, "adding 1")):
    z = sdata.table(cmp.data_file("dune"))
    quiet(z.add_log_ratio_columns, DUNE, **replaced)
    cmp.check_matrix("dune (vegan), 27 species", f'add_log_ratio_columns (zeros_replaced_by="{label}")', "clr",
                     Result(z).columns_of([f"clr_{s}" for s in DUNE]))

Read "Hydrochem.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="clr") | log ratios, column 1 [1]: simplydata -19.8933435549, R compositions clr (help page data) (compositions 2.0.9) -19.8933435549; relative difference 0.00e+00 (allowed 1e-10)
      Source: R compositions clr (help page data) (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/clr.html
agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="clr") | log ratios, column 1 [11]: simplydata -19.9288413285, R compositions clr (help page data) (compositions 2.0.9) -19.9288413285; relative difference 0.00e+00 (allowed 1e-10)
      Source: R compositions clr (help page data) (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/clr.html
agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="clr") | log ratios, column 1 [21]: simplydata -20.3278675912, R compositions clr (help page data) (compositions 

agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="alr", against="Cl") | log ratios, column 1 [1]: simplydata -24.4972511622, R compositions alr (compositions 2.0.9) -24.4972511622; relative difference 1.45e-16 (allowed 1e-10)
      Source: R compositions alr (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/alr.html
agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="alr", against="Cl") | log ratios, column 1 [11]: simplydata -24.5832977982, R compositions alr (compositions 2.0.9) -24.5832977982; relative difference 1.45e-16 (allowed 1e-10)
      Source: R compositions alr (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/alr.html
agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="alr", against="Cl") | log ratios, column 1 [21]: simplydata -24.9536742264, R compositions alr (compositions 2.0.9) -24.9536742264; relative difference 

Read "Hydrochem.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="ilr") | log ratios, column 1 [1]: simplydata 16.772462187, R compositions ilr (compositions 2.0.9) 16.772462187; relative difference 2.12e-16 (allowed 1e-10)
      Source: R compositions ilr (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/ilr.html
agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="ilr") | log ratios, column 1 [11]: simplydata 17.0623145471, R compositions ilr (compositions 2.0.9) 17.0623145471; relative difference 2.08e-16 (allowed 1e-10)
      Source: R compositions ilr (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/ilr.html
agrees  Hydrochem (compositions) | add_log_ratio_columns (test_method="ilr") | log ratios, column 1 [21]: simplydata 17.3980960745, R compositions ilr (compositions


agrees  dune (vegan), 27 species | add_log_ratio_columns (zeros_replaced_by="estimated small values") | clr, column 7 [4]: simplydata 1.08920792665, R zCompositions cmultRepl (GBM) + compositions clr (zCompositions 1.6.2) 1.08920792665; relative difference 8.15e-16 (allowed 1e-10)
      Source: R zCompositions cmultRepl (GBM) + compositions clr (zCompositions 1.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/zCompositions/html/cmultRepl.html
agrees  dune (vegan), 27 species | add_log_ratio_columns (zeros_replaced_by="estimated small values") | clr, column 7 [5]: simplydata 0.530411685072, R zCompositions cmultRepl (GBM) + compositions clr (zCompositions 1.6.2) 0.530411685072; relative difference 2.09e-16 (allowed 1e-10)
      Source: R zCompositions cmultRepl (GBM) + compositions clr (zCompositions 1.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/zCompositions/html/cmultRepl.html
agrees  dune (vegan), 27 species | add_log_ratio_columns (zeros_replace

agrees  dune (vegan), 27 species | add_log_ratio_columns (zeros_replaced_by="adding 1") | clr, column 1 [1]: simplydata 0.430551742198, R compositions clr of the counts plus 1 (compositions 2.0.9) 0.430551742198; relative difference 1.29e-16 (allowed 1e-10)
      Source: R compositions clr of the counts plus 1 (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/clr.html
agrees  dune (vegan), 27 species | add_log_ratio_columns (zeros_replaced_by="adding 1") | clr, column 1 [2]: simplydata 0.787989372321, R compositions clr of the counts plus 1 (compositions 2.0.9) 0.787989372321; relative difference 2.82e-16 (allowed 1e-10)
      Source: R compositions clr of the counts plus 1 (compositions 2.0.9) (run reference) - https://search.r-project.org/CRAN/refmans/compositions/html/clr.html
agrees  dune (vegan), 27 species | add_log_ratio_columns (zeros_replaced_by="adding 1") | clr, column 1 [3]: simplydata -0.569494306984, R compositions clr of t

## 7. Explaining profiles: db-RDA, RDA on Hellinger shares, CCA (the `vegan::dbrda` help-page data)

In [8]:
both = sdata.table(cmp.data_file("varespec_and_varechem"))
CASES = [({}, "bray-curtis (default)", ((999, 1), (499, 3))), ({"allowing_for": ["Al"]}, "bray-curtis, allowing_for Al", ((999, 1),)),
         ({"measure": "hellinger"}, "hellinger", ((999, 1),)), ({"test_method": "canonical correspondence"}, "canonical correspondence", ((999, 1),))]
for options, label, runs in CASES:
    for permutations, seed in runs:
        e = Result(quiet(both.explain_profiles_of, SPECIES, using=["N", "P", "K"], permutations=permutations, with_seed=seed, **options), key="term")
        check = f"explain_profiles_of ({label}, permutations={permutations}, with_seed={seed})"
        ours = {"all together: pseudo-F": e["all together", "pseudo-F"], "R squared": e["all together", "R squared"],
                "adjusted R squared": e["all together", "adjusted R squared"], "all together: p-value": e["all together", "p-value"]}
        for column in ("N", "P", "K"):
            ours[f"{column} after the others: pseudo-F"] = e[column, "pseudo-F"]
            ours[f"{column} after the others: p-value"] = e[column, "p-value"]
        cmp.check_all("varespec and varechem (vegan)", check, ours)

Read "varespec_and_varechem.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis (default), permutations=999, with_seed=1) | all together: pseudo-F: simplydata 1.6243107461, R vegan anova and RsquareAdj (vegan 2.7.8) 1.6243107461; relative difference 1.37e-16 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis (default), permutations=999, with_seed=1) | R squared: simplydata 0.195913058887, R vegan anova and RsquareAdj (vegan 2.7.8) 0.195913058887; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis (default), permut

agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis (default), permutations=499, with_seed=3) | all together: pseudo-F: simplydata 1.6243107461, R vegan anova and RsquareAdj (vegan 2.7.8) 1.6243107461; relative difference 1.37e-16 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis (default), permutations=499, with_seed=3) | R squared: simplydata 0.195913058887, R vegan anova and RsquareAdj (vegan 2.7.8) 0.195913058887; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis (default), permutations=499, with_seed=3) | adjusted R squared: simplydata 0.0753000177198, R vegan RsquareAdj (vegan 2.7.

agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis, allowing_for Al, permutations=999, with_seed=1) | all together: pseudo-F: simplydata 2.37167973302, R vegan anova and RsquareAdj (vegan 2.7.8) 2.37167973302; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis, allowing_for Al, permutations=999, with_seed=1) | R squared: simplydata 0.214137737089, R vegan anova and RsquareAdj (vegan 2.7.8) 0.214137737089; relative difference 2.59e-16 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (bray-curtis, allowing_for Al, permutations=999, with_seed=1) | adjusted R squared: simplydata 0.129477720116, R vegan 

agrees  varespec and varechem (vegan) | explain_profiles_of (hellinger, permutations=999, with_seed=1) | all together: pseudo-F: simplydata 1.68911219341, R vegan anova and RsquareAdj (vegan 2.7.8) 1.68911219341; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (hellinger, permutations=999, with_seed=1) | R squared: simplydata 0.202148982363, R vegan anova and RsquareAdj (vegan 2.7.8) 0.202148982363; relative difference 1.37e-16 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (hellinger, permutations=999, with_seed=1) | adjusted R squared: simplydata 0.0824713297171, R vegan RsquareAdj (vegan 2.7.8) 0.0824713297171; relative diffe

agrees  varespec and varechem (vegan) | explain_profiles_of (canonical correspondence, permutations=999, with_seed=1) | all together: pseudo-F: simplydata 1.81818012918, R vegan anova and RsquareAdj (vegan 2.7.8) 1.81818012918; relative difference 1.22e-16 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (canonical correspondence, permutations=999, with_seed=1) | R squared: simplydata 0.214285557881, R vegan anova and RsquareAdj (vegan 2.7.8) 0.214285557881; relative difference 1.30e-16 (allowed 1e-10)
      Source: R vegan anova and RsquareAdj (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/dbrda.html
agrees  varespec and varechem (vegan) | explain_profiles_of (canonical correspondence, permutations=999, with_seed=1) | adjusted R squared: simplydata 0.0960980387119, R vegan RsquareAdj 

## 8. Comparing and correlating profiles: PERMANOVA (Aitchison), SIMPER (the help-page example), Mantel, PROTEST

In [9]:
dd = sdata.table(cmp.data_file("dune_and_env"))
cp = Result(quiet(dd.compare_profiles_of, DUNE, for_each_group_in="Management", measure="aitchison")).row(1)
cmp.check_all("dune and dune.env (vegan)", 'compare_profiles_of (measure="aitchison", permutations=999, with_seed=1)',
              {"pseudo-F": cp["pseudo-F"] if "pseudo-F" in cp else cp["statistic"], "R squared": cp["R squared"], "p-value": cp["p-value"]})
simper = Result(quiet(dd.compare_profiles_of, DUNE_ALL, for_each_group_in="Management", show="contributions"))
check = 'compare_profiles_of (show="contributions")'
for first, second, species, value in zip(simper["first group"], simper["second group"], simper["column"], simper["average contribution"]):
    key = f"{first}_{second}: {species}" if cmp.has("dune and dune.env (vegan)", check, f"{first}_{second}: {species}") else f"{second}_{first}: {species}"
    cmp.check("dune and dune.env (vegan)", check, key, value)
for test_method, permutations, seed in (("mantel", 999, 1), ("mantel", 499, 4), ("procrustes", 999, 1)):
    fc = Result(quiet(both.find_correlation_between_profiles, first=SPECIES, second=["N", "P", "K"], test_method=test_method,
                      permutations=permutations, with_seed=seed)).row(1)
    cmp.check_all("varespec and varechem (vegan)", f'find_correlation_between_profiles (test_method="{test_method}", permutations={permutations}, with_seed={seed})',
                  {"statistic": fc["statistic"], "p-value": fc["p-value"]})

Read "dune_and_env.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  dune and dune.env (vegan) | compare_profiles_of (measure="aitchison", permutations=999, with_seed=1) | pseudo-F: simplydata 2.19995033686, R vegan adonis2 on the same pipeline (vegan 2.7.8) 2.19995033686; relative difference 2.02e-16 (allowed 1e-10)
      Source: R vegan adonis2 on the same pipeline (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/adonis2.html
agrees  dune and dune.env (vegan) | compare_profiles_of (measure="aitchison", permutations=999, with_seed=1) | R squared: simplydata 0.292030730976, R vegan adonis2 on the same pipeline (vegan 2.7.8) 0.292030730976; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan adonis2 on the same pipeline (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/adonis2.html
agrees  dune and dune.env (vegan) | compare_profiles_of (measure="aitchison", permutations=999, with_seed=1) | p-value: simplydata 0.002, R vegan adonis2 (same seed), 4 MCSE (vegan 2.

agrees  dune and dune.env (vegan) | compare_profiles_of (show="contributions") | SF_BF: Vicilath: simplydata 0.0146712547562, R vegan simper (help page example) (vegan 2.7.8) 0.0146712547562; relative difference 2.13e-15 (allowed 1e-10)
      Source: R vegan simper (help page example) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/simper.html
agrees  dune and dune.env (vegan) | compare_profiles_of (show="contributions") | SF_BF: Hyporadi: simplydata 0.0102856117533, R vegan simper (help page example) (vegan 2.7.8) 0.0102856117533; relative difference 4.39e-15 (allowed 1e-10)
      Source: R vegan simper (help page example) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/simper.html
agrees  dune and dune.env (vegan) | compare_profiles_of (show="contributions") | SF_BF: Ranuflam: simplydata 0.00930574298995, R vegan simper (help page example) (vegan 2.7.8) 0.00930574298995; relative difference 1.86e-15 (allowed 1e

agrees  dune and dune.env (vegan) | compare_profiles_of (show="contributions") | SF_HF: Poatriv: simplydata 0.02846625553, R vegan simper (help page example) (vegan 2.7.8) 0.02846625553; relative difference 2.44e-16 (allowed 1e-10)
      Source: R vegan simper (help page example) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/simper.html
agrees  dune and dune.env (vegan) | compare_profiles_of (show="contributions") | SF_HF: Bracruta: simplydata 0.0252608093613, R vegan simper (help page example) (vegan 2.7.8) 0.0252608093613; relative difference 3.85e-15 (allowed 1e-10)
      Source: R vegan simper (help page example) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/simper.html
agrees  dune and dune.env (vegan) | compare_profiles_of (show="contributions") | SF_HF: Eleopalu: simplydata 0.0249743670985, R vegan simper (help page example) (vegan 2.7.8) 0.0249743670985; relative difference 2.78e-16 (allowed 1e-10)
  

agrees  varespec and varechem (vegan) | find_correlation_between_profiles (test_method="mantel", permutations=999, with_seed=1) | statistic: simplydata 0.247589394555, R vegan mantel (vegan 2.7.8) 0.247589394555; relative difference 2.24e-16 (allowed 1e-10)
      Source: R vegan mantel (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/mantel.html
agrees  varespec and varechem (vegan) | find_correlation_between_profiles (test_method="mantel", permutations=999, with_seed=1) | p-value: simplydata 0.007, R vegan (same seed), 4 MCSE (vegan 2.7.8) 0.007; difference 0.00e+00 (allowed 1e-02)
      Source: R vegan (same seed), 4 MCSE (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/mantel.html


agrees  varespec and varechem (vegan) | find_correlation_between_profiles (test_method="mantel", permutations=499, with_seed=4) | statistic: simplydata 0.247589394555, R vegan mantel (vegan 2.7.8) 0.247589394555; relative difference 2.24e-16 (allowed 1e-10)
      Source: R vegan mantel (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/mantel.html
agrees  varespec and varechem (vegan) | find_correlation_between_profiles (test_method="mantel", permutations=499, with_seed=4) | p-value: simplydata 0.006, R vegan (same seed), 4 MCSE (vegan 2.7.8) 0.006; difference 0.00e+00 (allowed 1e-02)
      Source: R vegan (same seed), 4 MCSE (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/mantel.html
agrees  varespec and varechem (vegan) | find_correlation_between_profiles (test_method="procrustes", permutations=999, with_seed=1) | statistic: simplydata 0.445029146196, R vegan protest (vegan 2.7.8) 0.445029146196; relative differen

## 9. Hidden factors: exploratory factor analysis (Holzinger & Swineford) and the political democracy SEM

`find_hidden_factors_behind(how_many=3)` with minres/oblimin and ml/varimax against R `psych::fa`;
`check_hidden_factors(formula=)` against Rosseel's (2012) printed fit (chi-square 38.125 on 35 df, p = 0.329).

In [10]:
hs = sdata.table(FETCHED + "holzinger_swineford_1939.csv", separator=",")
TESTS = [f"x{i}" for i in range(1, 10)]
for test_method, rotation in (("minres", "oblimin"), ("ml", "varimax")):
    fa = Result(quiet(hs.find_hidden_factors_behind, TESTS, how_many=3, test_method=test_method, rotation=rotation))
    check = f'find_hidden_factors_behind (test_method="{test_method}", rotation="{rotation}", how_many=3)'
    cmp.check_matrix("Holzinger & Swineford (1939)", check, "loadings (factors matched)", fa.columns_of(["factor 1", "factor 2", "factor 3"]))
    cmp.check_vector("Holzinger & Swineford (1939)", check, "own part (uniqueness)", fa["own part"])
MODEL = ("ind60 =~ x1 + x2 + x3; dem60 =~ y1 + y2 + y3 + y4; dem65 =~ y5 + y6 + y7 + y8; dem60 ~ ind60; "
         "dem65 ~ ind60 + dem60; y1 ~~ y5; y2 ~~ y4 + y6; y3 ~~ y7; y4 ~~ y8; y6 ~~ y8")
sem = Result(quiet(sdata.table(cmp.data_file("PoliticalDemocracy")).check_hidden_factors, formula=MODEL), key="term")
chi = [t for t in sem.index if "chi-square" in t.lower()][0]
cmp.check_all("PoliticalDemocracy (lavaan)", "check_hidden_factors (formula=)",
              {"chi-square": sem[chi, "estimate"], "degrees of freedom": sem[chi, "degrees of freedom"], "p-value": sem[chi, "p-value"]})

explained difference – precision limit (B24)  Holzinger & Swineford (1939) | find_hidden_factors_behind (test_method="minres", rotation="oblimin", how_many=3) | loadings (factors matched), column 1 [1]: simplydata 0.196439133949, R psych fa (minres, oblimin) (psych 2.6.9) 0.196438911686; relative difference 1.13e-06 (allowed 1e-06)


      why: Oblimin rotation converges to GPArotation's tolerance - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b24-oblimin-rotation-converges-to-gparotations-tolerance
      Source: R psych fa (minres, oblimin) (psych 2.6.9) (run reference) - https://search.r-project.org/CRAN/refmans/psych/html/fa.html
explained difference – precision limit (B24)  Holzinger & Swineford (1939) | find_hidden_factors_behind (test_method="minres", rotation="oblimin", how_many=3) | loadings (factors matched), column 1 [2]: simplydata 0.042971371546, R psych fa (minres, oblimin) (psych 2.6.9) 0.0429709316601; relative difference 1.02e-05 (allowed 1e-06)
      why: Oblimin rotation converges to GPArotation's tolerance - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b24-oblimin-rotation-converges-to-gparotations-tolerance
      Source: R psych fa (minres, oblimin) (psych 2.6.9) (run reference) - https://search.r-project.org/CRAN/refmans/psych/html/fa.html


agrees  PoliticalDemocracy (lavaan) | check_hidden_factors (formula=) | chi-square: simplydata 38.1252182281, Rosseel (2012), lavaan: An R Package for Structural Equation Modeling, J. Stat. Softw. 48(2), section 5 38.125; difference 2.18e-04 (allowed 5e-04)
      Source: Rosseel (2012), lavaan: An R Package for Structural Equation Modeling, J. Stat. Softw. 48(2), section 5 (printed) - https://doi.org/10.18637/jss.v048.i02
agrees  PoliticalDemocracy (lavaan) | check_hidden_factors (formula=) | degrees of freedom: simplydata 35, Rosseel (2012), lavaan: An R Package for Structural Equation Modeling, J. Stat. Softw. 48(2), section 5 35; difference 0.00e+00 (allowed 5e-01)
      Source: Rosseel (2012), lavaan: An R Package for Structural Equation Modeling, J. Stat. Softw. 48(2), section 5 (printed) - https://doi.org/10.18637/jss.v048.i02
agrees  PoliticalDemocracy (lavaan) | check_hidden_factors (formula=) | p-value: simplydata 0.329180353225, Rosseel (2012), lavaan: An R Package for Struct

## 10. Chao1: BCI (the help-page data of `vegan::estimateR`)

In [11]:
bci = sdata.table(cmp.data_file("BCI"))
quiet(bci.add_diversity_columns, list(quiet(bci.get_column_names)), show="chao1")
chao = Result(bci)
cmp.check_vector("BCI (vegan)", 'add_diversity_columns (show="chao1")', "Chao1", chao["chao1"])
cmp.check_vector("BCI (vegan)", 'add_diversity_columns (show="chao1")', "Chao1 standard error", chao["chao1 standard error"])

Read "BCI.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  BCI (vegan) | add_diversity_columns (show="chao1") | Chao1 [1]: simplydata 117.473684211, R vegan estimateR (help page data) (vegan 2.7.8) 117.473684211; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan estimateR (help page data) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/estimateR.html
agrees  BCI (vegan) | add_diversity_columns (show="chao1") | Chao1 [2]: simplydata 117.214285714, R vegan estimateR (help page data) (vegan 2.7.8) 117.214285714; relative difference 0.00e+00 (allowed 1e-10)
      Source: R vegan estimateR (help page data) (vegan 2.7.8) (run reference) - https://search.r-project.org/CRAN/refmans/vegan/html/estimateR.html
agrees  BCI (vegan) | add_diversity_columns (show="chao1") | Chao1 [3]: simplydata 141.230769231, R vegan estimateR (help page data) (vegan 2.7.8) 141.230769231; relative difference 0.00e+00 (allowed 1

In [12]:
cmp.finish()

28_patterns_across_columns: 5712 comparisons - 5690 agrees, 22 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,28_patterns_across_columns,USArrests,find_principal_components_of,standard deviations [1],1.574878,1.574878,R stats prcomp (help page example) (stats 4.6.2),-4.440892e-16,1.000000e-10,relative,2.819832e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,28_patterns_across_columns,USArrests,find_principal_components_of,standard deviations [2],0.994869,0.994869,R stats prcomp (help page example) (stats 4.6.2),2.220446e-16,1.000000e-10,relative,2.231897e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,28_patterns_across_columns,USArrests,find_principal_components_of,standard deviations [3],0.597129,0.597129,R stats prcomp (help page example) (stats 4.6.2),-2.220446e-16,1.000000e-10,relative,3.718536e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,28_patterns_across_columns,USArrests,find_principal_components_of,standard deviations [4],0.416449,0.416449,R stats prcomp (help page example) (stats 4.6.2),1.665335e-16,1.000000e-10,relative,3.998888e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,28_patterns_across_columns,USArrests,find_principal_components_of,spread explained (%) [1],62.006039,62.006039,R stats prcomp (help page example) (stats 4.6.2),-2.131628e-14,1.000000e-10,relative,3.437775e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5707,28_patterns_across_columns,BCI (vegan),"add_diversity_columns (show=""chao1"")",Chao1 standard error [46],11.083109,11.083109,R vegan estimateR (help page data) (vegan 2.7.8),0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/vega...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
5708,28_patterns_across_columns,BCI (vegan),"add_diversity_columns (show=""chao1"")",Chao1 standard error [47],27.069508,27.069508,R vegan estimateR (help page data) (vegan 2.7.8),0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/vega...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
5709,28_patterns_across_columns,BCI (vegan),"add_diversity_columns (show=""chao1"")",Chao1 standard error [48],12.165478,12.165478,R vegan estimateR (help page data) (vegan 2.7.8),0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/vega...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
5710,28_patterns_across_columns,BCI (vegan),"add_diversity_columns (show=""chao1"")",Chao1 standard error [49],15.707015,15.707015,R vegan estimateR (help page data) (vegan 2.7.8),0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/vega...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
